# 3 - External validation

Tests the signature (Table 2, from notebook 2) on GSE267650, a mouse series that took no
part in its construction (Section 4.6): non-operated controls, ischemia without
reperfusion, and animals reperfused for 4-72 h. Two comparisons are scored: reperfused vs
non-operated controls (`vs_controls`) and reperfused vs ischemia-only (`vs_ischemia`).

All numbers come from one limma fit per comparison, on log2 counts per million:
concordance, median oriented log2 fold change, and the permutation test of the mean
oriented moderated t (1,000 random reassignments of the group labels).

| Output | Use |
|---|---|
| `results/validation_summary.csv` | the numbers of Section 2.4 |
| `results/validation_genes_vs_controls.csv`, `results/validation_genes_vs_ischemia.csv` | one row per signature gene |
| `results/validation_permutations_vs_controls.csv`, `results/validation_permutations_vs_ischemia.csv` | the permutation statistics (row `perm = 0` holds the observed values) |
| `results/validation_design.csv` | groups and numbers of animals |
| `results/TableS5_GSE267650_samples.csv`, `results/TableS5_GSE267650_groups.csv` | Table S5: every sample with its GEO accession and group, and the animals per group and time point |
| `results/TableS6_concordance_by_direction.csv` | Table S6: concordance by direction of the signature genes |

In [1]:
# Move to the project root (the folder that contains run_all.ps1), so that relative
# paths work whether the notebook is started from notebook/ or from the root.
root <- normalizePath(getwd(), winslash = "/", mustWork = TRUE)
while (!file.exists(file.path(root, "run_all.ps1")) && dirname(root) != root) {
  root <- dirname(root)
}
if (!file.exists(file.path(root, "run_all.ps1"))) {
  stop("project root not found: no run_all.ps1 above ", getwd())
}
setwd(root)
.libPaths(c(file.path(root, "env", "rlib"), .libPaths()))
Sys.setenv(RENAL_IRI_ROOT = root)
options(stringsAsFactors = FALSE, warn = 1)
cat(R.version.string, "\nroot: ", root, "\n", sep = "")

R version 4.4.3 (2025-02-28)
root: /Users/robertocasale/Desktop/lavoro_FBF/renal_shield_perfusion


## Validation

`R/validation_GSE267650.R`; the permutation test takes a few minutes.

In [2]:
source("R/validation_GSE267650.R")

[2026-10-08 22:10:34] INFO  WROTE       results/validation_design.csv (4 rows)



[2026-10-08 22:10:43] INFO    Mus musculus: 451 outdated rodent symbols updated before ortholog mapping



[2026-10-08 22:10:53] INFO  WROTE       results/validation_genes_vs_controls.csv (256 rows)



[2026-10-08 22:13:03] INFO    Mus musculus: 450 outdated rodent symbols updated before ortholog mapping



[2026-10-08 22:13:05] INFO  WROTE       results/validation_genes_vs_ischemia.csv (256 rows)



[2026-10-08 22:15:16] INFO  WROTE       results/validation_summary.csv (2 rows)



  comparison n_reference n_reperfused signature_genes_measured concordant
 vs_controls           5           31                      256        249
 vs_ischemia           5           31                      256        241
 concordance median_oriented_logFC mean_t permutations r        p
      0.9727                 1.573  6.600         1000 0 0.000999
      0.9414                 1.293  5.669         1000 0 0.000999


## Samples of GSE267650 (Table S5)

Every sample of GSE267650 with its GEO accession, the genotype and time point of its GEO
sample record, and the group it has in the validation above: the 41 wild-type animals are
analysed and the 16 genetically modified animals are excluded. The GEO records do not give
the sex or the age of the mice; the original study (Heruye et al., J Clin Invest 2024,
reference 35 of the article) used male C57BL/6J mice aged 8-12 weeks.

In [3]:
# Table S5: every sample of GSE267650 with its GEO accession, and the group it has in the
# validation above. The GEO sample records are read from the series matrix file of
# GSE267650, a text file with one line per field ("!Sample_title", "!Sample_geo_accession",
# one "!Sample_characteristics_ch1" line per characteristic) and one column per sample.
# The table is downloaded once and cached like every other GEO answer.
geo <- robust_fetch("geo_samples_GSE267650", fn = function() {
  f <- file.path(DIRS$holdout, "GSE267650_series_matrix.txt.gz")
  utils::download.file(paste0("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE267nnn/GSE267650/",
                              "matrix/GSE267650_series_matrix.txt.gz"), f, mode = "wb", quiet = TRUE)
  lines  <- readLines(gzfile(f))
  values <- function(field) lapply(grep(paste0("^!Sample_", field, "\t"), lines, value = TRUE),
                                   function(l) gsub('"', "", strsplit(l, "\t")[[1]][-1]))
  out <- data.frame(gsm = values("geo_accession")[[1]], title = values("title")[[1]])
  for (v in values("characteristics_ch1")) {          # "tissue: kidney", "time: 4h", ...
    name <- unique(sub(":.*$", "", v))
    stopifnot(length(name) == 1)
    out[[name]] <- sub("^[^:]*: ", "", v)
  }
  out
})
stopifnot("GSE267650 sample records not available" = geo$ok)
geo <- geo$value

# 1. The column names of the counts file (used by the validation) and the GEO titles are
#    spelled differently in a few samples (for example WT_Ctl-No-IR_1 and WT_Ctl-No-IRI_1,
#    Azin1_locked_IRI_48_1 and Azin1_locked_IRI_48h_1). Both are therefore reduced to
#    genotype | time | replicate number and matched on that key.
replicate_of <- function(x) sub("^.*_([0-9]+)$", "\\1", x)
geo$key <- paste(geo$genotype, geo$time, replicate_of(geo$title), sep = "|")
columns <- colnames(raw)[-(1:3)]
col_genotype <- ifelse(grepl("^WT_", columns), "WT",
                ifelse(grepl("^Azin1_locked_", columns), "Azin1_locked",
                ifelse(grepl("^(Azin1_uneditable|AZIN1_unedit)_", columns), "Azin1_unedit", NA)))
col_time <- ifelse(grepl("_Ctl-No-IR_", columns), "none",
            ifelse(grepl("_Ischemia_20min_without_reperfusion_", columns), "20min",
                   sub("^.*_([0-9]+)h?_[0-9]+$", "\\1h", columns)))
col_key <- paste(col_genotype, col_time, replicate_of(columns), sep = "|")
stopifnot(nrow(geo) == length(columns), all(geo$tissue == "kidney"),
          !anyDuplicated(geo$key), !anyDuplicated(col_key), setequal(geo$key, col_key))

# 2. One row per sample, in the order of the counts file, with the group of the validation:
#    the 41 wild-type animals in `cnt` and their labels `g` (cell above); the other samples
#    are the genetically modified animals excluded there.
s5 <- geo[match(col_key, geo$key), c("gsm", "title", "genotype", "treatment", "time")]
names(s5)[names(s5) == "title"] <- "geo_title"
s5$counts_column <- columns
label <- setNames(g, colnames(cnt))[columns]          # NA for the excluded samples
s5$group <- ifelse(is.na(label), "excluded (genetically modified)",
            ifelse(label == CTL, "non-operated control",
            ifelse(label == ISCH, "ischemia only",
            ifelse(label %in% TIMES, "reperfused", NA))))
s5$hours_after_reperfusion <- NA_integer_
s5$hours_after_reperfusion[s5$group == "reperfused"] <-
  as.integer(sub("h$", "", s5$time[s5$group == "reperfused"]))
groups <- c("non-operated control", "ischemia only", "reperfused",
            "excluded (genetically modified)")
reperfused <- s5$group == "reperfused"
stopifnot(!anyNA(s5$group),
          all(s5$genotype[s5$group != groups[4]] == "WT"),       # analysed: wild type only
          all(s5$genotype[s5$group == groups[4]] != "WT"),
          all(s5$treatment[s5$group == groups[1]] == "none"),
          all(s5$time[s5$group == groups[1]] == "none"),
          all(s5$time[s5$group == groups[2]] == "20min"),
          all(s5$treatment[s5$group %in% groups[2:3]] == "IRI"),
          # the time in GEO is the time of the label used by the validation
          identical(paste0("WT_IRI_", s5$time[reperfused]), unname(label[reperfused])),
          # the same numbers of animals as results/validation_design.csv
          identical(as.integer(table(factor(s5$group, levels = groups))), as.integer(design$n)))
save_csv(s5[, c("gsm", "geo_title", "counts_column", "genotype", "treatment", "time",
                "group", "hours_after_reperfusion")], "TableS5_GSE267650_samples.csv")

# 3. The animals of each group and time point, with the range of their GEO accessions
#    (the code stops if the accessions of a row are not consecutive).
rows <- unique(s5[, c("group", "hours_after_reperfusion")])
rows <- rows[order(match(rows$group, groups), rows$hours_after_reperfusion), ]
s5_groups <- do.call(rbind, lapply(seq_len(nrow(rows)), function(i) {
  k <- s5$group == rows$group[i] &
       (s5$hours_after_reperfusion %in% rows$hours_after_reperfusion[i])
  n <- sort(as.integer(sub("^GSM", "", s5$gsm[k])))
  stopifnot(all(diff(n) == 1))
  data.frame(group = rows$group[i], hours_after_reperfusion = rows$hours_after_reperfusion[i],
             animals = sum(k), gsm_first = paste0("GSM", n[1]), gsm_last = paste0("GSM", n[length(n)]))
}))
stopifnot(sum(s5_groups$animals) == nrow(s5))
save_csv(s5_groups, "TableS5_GSE267650_groups.csv")
print(s5_groups, row.names = FALSE)

[2026-10-08 22:15:17] INFO  LIVE OK     geo_samples_GSE267650 (attempt 1/4)



[2026-10-08 22:15:17] INFO  WROTE       results/TableS5_GSE267650_samples.csv (57 rows)



[2026-10-08 22:15:17] INFO  WROTE       results/TableS5_GSE267650_groups.csv (9 rows)



                           group hours_after_reperfusion animals  gsm_first
            non-operated control                      NA       5 GSM8271629
                   ischemia only                      NA       5 GSM8271634
                      reperfused                       4       5 GSM8271639
                      reperfused                      16       5 GSM8271644
                      reperfused                      24       5 GSM8271649
                      reperfused                      36       5 GSM8271654
                      reperfused                      48       5 GSM8271659
                      reperfused                      72       6 GSM8271664
 excluded (genetically modified)                      NA      16 GSM8271670
   gsm_last
 GSM8271633
 GSM8271638
 GSM8271643
 GSM8271648
 GSM8271653
 GSM8271658
 GSM8271663
 GSM8271669
 GSM8271685


## Concordance by direction (Table S6)

Concordance counted separately for the 237 signature genes that go up and the 19 that go
down, because the total is dominated by the upregulated genes. The cell also checks that
every signature gene has a finite, non-zero log2 fold change in both comparisons.

In [4]:
# Table S6: concordance in GSE267650 counted separately for the signature genes that go up
# and for those that go down, in each comparison. It reads the gene-level tables written
# above and checks that every signature gene has a finite, non-zero log2 fold change.
sig  <- read.csv("results/Table2_signature.csv")
sig  <- sig[sig$signature, c("human", "direction")]
summ <- read.csv("results/validation_summary.csv")
s6 <- do.call(rbind, lapply(c("vs_controls", "vs_ischemia"), function(cmp) {
  g <- read.csv(sprintf("results/validation_genes_%s.csv", cmp))
  stopifnot(nrow(g) == nrow(sig), !anyDuplicated(g$human), setequal(g$human, sig$human),
            all(g$direction %in% c("up", "down")),
            identical(g$direction, sig$direction[match(g$human, sig$human)]),
            all(is.finite(g$logFC)), all(g$logFC != 0),
            identical(g$concordant, sign(g$logFC) == ifelse(g$direction == "up", 1, -1)),
            identical(sum(g$concordant), summ$concordant[summ$comparison == cmp]))
  do.call(rbind, lapply(c("up", "down", "all"), function(d) {
    k <- if (d == "all") rep(TRUE, nrow(g)) else g$direction == d
    data.frame(comparison = cmp, direction = d, genes = sum(k),
               concordant = sum(g$concordant[k]), concordance = mean(g$concordant[k]))
  }))
}))
save_csv(s6, "TableS6_concordance_by_direction.csv")
cat("All", nrow(sig), "signature genes have a finite, non-zero log2 fold change in both comparisons.\n")
print(s6, row.names = FALSE, digits = 4)

[2026-10-08 22:15:18] INFO  WROTE       results/TableS6_concordance_by_direction.csv (6 rows)



All 256 signature genes have a finite, non-zero log2 fold change in both comparisons.


  comparison direction genes concordant concordance
 vs_controls        up   237        231      0.9747
 vs_controls      down    19         18      0.9474
 vs_controls       all   256        249      0.9727
 vs_ischemia        up   237        223      0.9409
 vs_ischemia      down    19         18      0.9474
 vs_ischemia       all   256        241      0.9414
